# 외국인 유학생 중도탈락 예측 — 로지스틱 회귀

**강의용 노트북 · 데이터 분석 입문자 대상**

이 노트북은 대학의 외국인 유학생 학적 데이터로 **중도탈락(제적) 위험 학생을 미리 찾아내는 모델**을 만듭니다.
같은 데이터를 XGBoost로 다루는 노트북이 따로 있습니다. 두 노트북을 나란히 보면
**"모델이 달라지면 전처리가 어떻게 달라지는가"** 를 알 수 있습니다.

---

## 이 노트북에서 배우는 것

| # | 주제 | 왜 중요한가 |
|---|------|------------|
| 1 | 원천 데이터 감사 | 코드를 쓰기 전에 데이터를 먼저 세어봐야 합니다 |
| 2 | 로지스틱 회귀 전용 전처리 | 더미화·스케일링·결측 대체가 왜 필요한지 |
| 3 | 파이프라인 구성 | 데이터 누수를 구조적으로 막는 방법 |
| 4 | 다중공선성 진단 | 변수끼리 겹치면 계수를 믿을 수 없습니다 |
| 5 | 오즈비 해석 | "이 변수가 위험을 몇 배 올리나"를 읽는 법 |
| 6 | 불균형 데이터 다루기 | 정확도 95%가 쓸모없을 수 있습니다 |
| 7 | 임계값 최적화 | 몇 점부터 "위험군"으로 볼 것인가 |
| 8 | SHAP 요인 분석 | 모델이 무엇을 보고 판단했는지 |
| 9 | 위험군 명단 산출 | 현장 상담으로 이어지는 결과물 |

---

## 예측의 구조

```
[결과가 확정된 학생]                     [아직 결과를 모르는 학생]
 졸업·수료·제적·입학취소                        재학·휴학
        │                                        │
        │ 여기서 규칙을 배운다                     │ 배운 규칙을 적용한다
        ▼                                        ▼
    모델 학습  ─────────────────────────▶  위험군 명단
```

> **📌 지도학습 (supervised learning)**
>
> 정답을 아는 데이터로 규칙을 배우고, 정답을 모르는 데이터에 그 규칙을 적용하는 방식입니다.
> 여기서 "정답"은 **이 학생이 제적되었는가**이고, 졸업했거나 제적된 학생은 이미 답을 압니다.
> 지금 재학 중인 학생은 아직 답이 없으니, 그들이 예측 대상입니다.

## 1. 환경 준비

### 1-1. 라이브러리 불러오기

**무엇을 하나요**
분석에 쓸 도구들을 메모리에 올립니다.

**왜 필요한가요**
파이썬은 필요한 도구만 골라 쓰는 방식입니다. `import`하지 않은 기능은 존재하지 않는 것과 같습니다.

**결과를 어떻게 읽나요**
아래에 각 라이브러리 버전이 출력됩니다. 버전이 크게 다르면 코드 동작이 달라질 수 있으니 기록해 둡니다.

In [ ]:
import warnings
warnings.simplefilter('ignore')   # 강의 중 경고 메시지로 화면이 어지러워지는 것을 막습니다

# --- 데이터 다루기 ---
import numpy as np
import pandas as pd
from datetime import datetime

# --- 전처리 ---
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# --- 모델 ---
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- 검증 / 평가 ---
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             accuracy_score, roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve)

# --- 해석 / 시각화 ---
import shap
import matplotlib
import matplotlib.pyplot as plt

import sklearn
print('pandas     ', pd.__version__)
print('numpy      ', np.__version__)
print('scikit-learn', sklearn.__version__)
print('statsmodels', sm.__version__ if hasattr(sm, '__version__') else '-')
print('shap       ', shap.__version__)

### 1-2. 한글 폰트 설정

**무엇을 하나요**
그래프에 한글이 깨지지 않도록 폰트를 지정합니다.

**왜 필요한가요**
matplotlib의 기본 폰트에는 한글 글리프가 없습니다. 설정하지 않으면 `최종GPA`가 `□□□□`로 나옵니다.
변수명이 전부 한글인 이 데이터에서는 그래프가 통째로 쓸모없어집니다.

**결과를 어떻게 읽나요**
선택된 폰트 이름이 출력됩니다. `None`이 나오면 한글 폰트가 설치되어 있지 않은 것이니
`pip install koreanize-matplotlib` 등을 고려하세요.

In [ ]:
import matplotlib.font_manager as fm

In [ ]:
!apt-get -qq install fonts-nanum

In [ ]:
# 2. matplotlib 폰트 캐시 갱신
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')

# 3. 한글 폰트 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

> **📌 `axes.unicode_minus = False`**
>
> 한글 폰트에는 matplotlib이 기본으로 쓰는 유니코드 음수 기호(−, U+2212)가 없는 경우가 많습니다.
> 이 설정은 평범한 하이픈(-)을 쓰게 해서 **SHAP 값의 음수 부호가 깨지는 것**을 막습니다.
> 한글 폰트를 지정할 때 거의 항상 함께 써야 하는 짝입니다.

### 1-3. 설정값 모아두기

**무엇을 하나요**
파일 경로, 난수 시드처럼 나중에 바꿀 만한 값을 맨 위에 모읍니다.

**왜 필요한가요**
새 학기 데이터로 다시 돌릴 때 노트북 전체를 뒤지지 않고 이 셀 하나만 고치면 됩니다.
설정값이 코드 곳곳에 흩어져 있으면 한 군데를 빠뜨려 엉뚱한 결과가 나옵니다.

**결과를 어떻게 읽나요**
경로가 실제 파일을 가리키는지 확인하세요. `파일 없음`이 뜨면 여기서 멈추고 경로부터 고칩니다.

**Colab에서 실행**: 같은 CSV를 왼쪽 파일 패널에 올린 뒤 위에서부터 실행하세요.
아래 경로 셀은 `/content/외국인자료(202609).csv`를 먼저 찾고, 없으면 기존 Google Drive 경로를 사용합니다.
추가 진단은 **5-5. 학년별 제적률 확인 → 학년 제외 → p값 선별 → 최종 회귀**에 있습니다. 기존 저장 출력은 오래된 결과/개인정보 혼동을 피하기 위해 새 사본에서 비웠습니다.

In [ ]:
import os
from pathlib import Path
DATA_PATH = '/content/외국인자료(202609).csv'
if not Path(DATA_PATH).exists():
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/data/외국인자료(202609).csv'
if not Path(DATA_PATH).exists():
    raise FileNotFoundError('CSV를 Colab에 업로드하거나 DATA_PATH를 수정하세요.')
RANDOM_STATE = 1
TEST_SIZE = 0.3
YEARS_BACK = 10
CURRENT_YEAR = 2026  # 첨부된 2026년 9월 자료 기준. 새 자료 분석 시 수정하세요.
print('데이터 경로:', DATA_PATH)
print('분석 대상:', CURRENT_YEAR - YEARS_BACK, '년 이후 입학생')

---

## 2. 데이터 불러오기와 감사

### 2-1. 인코딩을 판별하며 읽기

**무엇을 하나요**
CSV 파일을 읽되, 인코딩을 여러 개 시도해서 성공하는 것을 씁니다.

**왜 필요한가요**
한글이 든 CSV는 만든 환경에 따라 `utf-8`, `cp949`, `euc-kr` 등으로 저장됩니다.
인코딩을 잘못 지정하면 `UnicodeDecodeError`가 나거나, 더 나쁘게는 **글자가 깨진 채로 읽힙니다**.
학교에서 매 학기 새 파일을 받는데 그때마다 인코딩이 다를 수 있으니 자동 판별이 안전합니다.

**결과를 어떻게 읽나요**
성공한 인코딩과 데이터 크기(행, 열)가 출력됩니다. 열 개수가 예상과 다르면 파일 양식이 바뀐 것입니다.

In [ ]:
DF = pd.read_csv(DATA_PATH, encoding="utf-8")

print(DF.shape)

print(type(DF.shape))

rows, cols = DF.shape

print(rows)
print(cols)
print(f'크기   : {DF.shape[0]:,}행 × {DF.shape[1]}열')
DF.head(3)

### 2-2. 어떤 열이 있고, 얼마나 비어 있나

**무엇을 하나요**
열 이름·자료형·결측 비율을 한 표로 정리합니다.

**왜 필요한가요**
**코드를 쓰기 전에 데이터를 먼저 세어봐야 합니다.**
결측이 78%인 변수를 모델에 넣으면, 채워 넣은 값이 대다수가 되어 사실상 상수가 됩니다.
그런 변수는 아무 정보도 주지 않으면서 계수만 하나 늘립니다.

**결과를 어떻게 읽나요**
`결측%` 열을 보세요. 50%를 넘는 변수는 기본적으로 쓰지 않습니다.
`고유값` 열이 크면(예: 28) 더미 변수가 27개 만들어진다는 뜻이니 주의가 필요합니다.

In [ ]:
DF.info()

In [ ]:
audit = pd.DataFrame({
    '자료형'  : DF.dtypes.astype(str),
    '결측수'  : DF.isna().sum(),
    '결측%'   : (DF.isna().mean() * 100).round(1),
    '고유값'  : DF.nunique(),
})
audit.sort_values('결측%', ascending=False)

위 표에서 눈에 띄는 것들입니다.

| 변수 | 결측 | 판단 |
|---|---|---|
| `합격순위`, `합격인원` | 약 68% | **사용하지 않습니다.** 결측을 채우면 값의 대부분이 같아져 정보가 없어집니다 |
| `수시/정시/기타` | 약 60% | 대학원생은 전량 결측입니다. **학부 필터로만** 씁니다 |
| `TOPIK급수` | 약 51% | 결측은 "미제출"이라는 정보이므로 0으로 채워 **학부에서만** 씁니다 |
| `소속 전공` | 약 27% | 모델에 넣지 않습니다 |
| `신청학점` | 약 7% | 학과 평균으로 채워 씁니다 |
| `최종GPA` | 약 4% | 중앙값으로 채워 씁니다 |

> **📌 결측(missing value)을 채운다는 것**
>
> 빈칸을 그냥 두면 계산이 안 되니 무언가로 메웁니다. 그런데 메운 값은 **관측된 사실이 아니라 추측**입니다.
> 빈칸이 적을 때는 문제가 적지만, 빈칸이 대부분이면 그 변수는 "추측으로 만든 변수"가 됩니다.
> 그래서 결측률을 먼저 봅니다.

### 2-3. 예측 대상을 정하기 — 무엇을 "중도탈락"이라 부를 것인가

**무엇을 하나요**
`최종변동구분`과 `학적상태`의 분포를 보고, 어떤 학생을 "탈락"으로 볼지 정합니다.

**왜 필요한가요**
**정답의 정의가 바뀌면 모델 전체가 바뀝니다.** 이건 코딩 문제가 아니라 기관이 결정할 정책 문제입니다.
분석자가 조용히 정하고 넘어가면 안 되고, 무엇을 골랐는지 반드시 밝혀야 합니다.

**결과를 어떻게 읽나요**
두 열의 숫자를 비교해 보세요. `학적상태`의 제적 인원이 `최종변동구분`보다 많습니다. 그 차이가 무엇인지가 핵심입니다.

In [ ]:
print('--- 최종변동구분 ---')
print(DF['최종변동구분'].value_counts().to_string())
print()
print('--- 학적상태 ---')
print(DF['학적상태'].value_counts().to_string())
print()
차이 = (DF['학적상태'] == '제적').sum() - (DF['최종변동구분'] == '제적').sum()
print(f"학적상태 '제적' - 최종변동구분 '제적' = {차이}명")
print(f"최종변동구분 '자퇴' 인원          = {(DF['최종변동구분'] == '자퇴').sum()}명")

숫자가 정확히 맞아떨어집니다. **`학적상태`의 제적에는 자퇴가 포함되어 있습니다.**

- `최종변동구분 == '제적'` : 479명 — 학교가 학적을 제거한 경우 (미등록, 학사경고 누적, 미복학 등)
- `학적상태 == '제적'` : 755명 — 위 479명 **+ 자퇴 276명**

어느 쪽이 "중도탈락"일까요? 대학알리미 공시 기준의 중도탈락률은 **자퇴를 포함**합니다.
그런데 이 노트북은 **`학적상태 == '제적'`만** 씁니다. 이유는 두 가지입니다.


정의를 바꾸고 싶다면 아래 셀의 `TARGET_RULE`만 바꾸면 노트북 전체가 따라갑니다.

> **📌 타깃 (target) / 종속변수**
>
> 모델이 맞히려는 대상입니다. 여기서는 `제적여부`이고 값은 0(아님) 또는 1(제적)입니다.
> 0과 1 중 하나를 맞히는 문제를 **이진 분류(binary classification)** 라고 합니다.

In [ ]:
# 타깃 정의 — 여기만 바꾸면 노트북 전체에 반영됩니다
TARGET_RULE = lambda d: (d['학적상태'] == '제적').astype(int)
# 자퇴까지 포함하려면 아래 줄로 교체하세요
# TARGET_RULE = lambda d: d['최종변동구분'].isin(['제적', '자퇴']).astype(int)

print('현재 정의로 본 전체 양성(제적) 인원:', TARGET_RULE(DF).sum(), '명')

---

## 3. 왜 전처리를 새로 하는가 — 로지스틱 회귀 vs XGBoost

같은 데이터인데 XGBoost 노트북과 전처리가 다릅니다. 모델이 데이터를 쓰는 방식이 다르기 때문입니다.

### 두 모델이 판단하는 방식

**XGBoost(결정 트리 계열)** 는 질문을 이어 붙여 판단합니다.

```
최종GPA < 1.5 ?
├─ 예 → 휴학학기수 >= 2 ?
│        ├─ 예 → 위험 높음
│        └─ 아니오 → 위험 보통
└─ 아니오 → 위험 낮음
```

**로지스틱 회귀**는 모든 변수를 **하나의 식**에 넣고 더합니다.

```
로그오즈 = b0 + b1×이수학기수 + b2×최종GPA + b3×휴학학기수 + ...
```

트리는 변수를 하나씩 따로 보지만, 회귀는 **전부를 같은 저울에 올려 합산**합니다.
바로 이 차이에서 전처리의 차이가 전부 나옵니다.

### 전처리 대조표

| 항목 | XGBoost | 로지스틱 회귀 | 왜 다른가 |
|---|---|---|---|
| 범주형 변수 | `astype('category')` 로 원본 유지 | **원핫 더미화 필수** | 선형식은 숫자만 더할 수 있습니다 |
| 기준 범주 | 없음 | **하나를 빼야 함** (`drop='first'`) | 더미를 전부 넣으면 합이 항상 1로 고정됩니다 |
| 스케일링 | 불필요 | **필수** | 계수 크기를 비교하려면 단위를 맞춰야 합니다 |
| 결측값 | 알고리즘이 알아서 처리 | **직접 채워야 함** | 빈칸이 있으면 덧셈이 안 됩니다 |
| 비선형 관계 | 자동으로 학습 | 직접 만들어야 함 | 직선 하나로만 설명합니다 |
| 변수 간 중복 | 상관없음 | **VIF로 점검 필수** | 계수가 불안정해지고 부호가 뒤집힙니다 |
| 범주가 많은 변수 | 최적 분할을 찾음 | 더미가 폭발 → 묶어야 함 | 28개 범주 = 27개 열 추가 |
| 불균형 데이터 | `scale_pos_weight` | `class_weight='balanced'` | 이름만 다르고 취지는 같습니다 |

> **📌 원핫 인코딩 (one-hot encoding)**
>
> `계열`처럼 값이 글자인 변수를 0/1 열 여러 개로 바꾸는 것입니다.
> `계열 = 공학` 한 명은 `계열_공학=1, 계열_예체능=0, 계열_인문사회=0`이 됩니다.
> 이렇게 해야 "공학이면 얼마를 더한다"는 계산이 가능해집니다.

> **📌 기준 범주와 더미 변수 함정**
>
> 계열이 4종류면 더미도 4개 만들고 싶지만, **3개만 만듭니다.**
> 4개를 다 넣으면 세 개가 0일 때 나머지 하나는 자동으로 1이 되므로, 네 번째 열은 앞의 세 열로 완전히 예측됩니다.
> 이러면 계산이 아예 불가능해집니다(행렬이 특이해집니다). 빠진 하나를 **기준 범주**라 하고,
> 나머지 계수는 전부 "기준 범주와 비교해서 얼마나 다른가"로 읽습니다.

#### ✅ 체크포인트

1. 트리 모델은 스케일링이 필요 없는데, 회귀는 왜 필요할까요?
2. 계열이 4종류인데 더미를 3개만 만드는 이유는?

<details><summary>답 보기</summary>

1. 트리는 "GPA가 1.5보다 작은가?"처럼 **순서만** 봅니다. 단위를 바꿔도 순서는 그대로라 결과가 같습니다.
   반면 회귀는 모든 변수를 곱해서 **더합니다.** 나이(20~40)와 GPA(0~4.5)를 그대로 더하면 나이가 훨씬 큰 값이라
   저울이 한쪽으로 기울고, 계수 크기를 서로 비교할 수도 없습니다.
2. 4개를 다 넣으면 네 열의 합이 항상 1이 되어, 한 열이 나머지로 완전히 계산됩니다.
   이런 상태에서는 계수를 구하는 방정식에 답이 하나로 정해지지 않습니다.

</details>

---

## 4. 전처리 — 파생변수 만들기



### 4-1. 공통 파생변수

**무엇을 하나요**
두 집단에 공통으로 필요한 변수를 만듭니다. 나이, 국적 묶음, TOPIK 등급, 그리고 정답인 `제적여부`입니다.

**왜 필요한가요**
원본에는 `생년월일`이 `19970810` 같은 숫자로만 있습니다. 모델에 이대로 넣으면
"1,997만 대 숫자"로 취급되어 아무 의미가 없습니다. 사람이 이해하는 **나이**로 바꿔야 합니다.
국적도 마찬가지입니다. 60개 나라를 그대로 쓰면 더미가 60개 생기는데, 한 나라에 5명뿐이면 학습이 안 됩니다.

**결과를 어떻게 읽나요**
함수만 정의하고 실행은 다음 셀에서 합니다. 여기서는 에러 없이 정의되는지만 확인합니다.

In [ ]:
def add_common_features(d):
    d = d.copy()   # 원본을 건드리지 않기 위해 복사본에서 작업합니다


    # 국적: 중국·베트남이 대다수이므로 3개 묶음으로 단순화합니다
    d['국적범주화'] = np.where(d['국적'].isin(['중국', '베트남']), d['국적'], '기타국가')

    # TOPIK: '4등급', '4단계'가 섞여 있으므로 앞 1자리 숫자만 씁니다. 결측은 0(미제출)
    topik_digit = d['TOPIK급수'].fillna('0').astype(str).str[:1]
    d['topik'] = pd.to_numeric(topik_digit, errors='coerce').fillna(0).astype(int)

    # 타깃
    d['제적여부'] = TARGET_RULE(d)

    # 제적연도 (제적자에게만 값을 남깁니다)
    d['제적연도'] = np.where(d['최종변동구분'] == '제적',
                           d['최종변동일자'].astype(str).str[:4], None)

    d['휴학학기수'] = d['휴학학기수'].fillna(0)
    d['학번'] = d['학번'].astype('int64').astype(str)   # 식별자는 문자로 (계산에 쓰이지 않도록)
    return d

print('공통 파생변수 함수 정의 완료')

### 4-3. 학부생 전처리

**무엇을 하나요**
학부생만 골라내고, 분석 대상 조건으로 걸러낸 뒤, 학부 전용 변수를 만듭니다.

**왜 필요한가요**
전체 13,090명을 다 쓰면 안 됩니다. 2016년 이전 입학생은 학사 제도가 달라 지금 학생을 예측하는 데 도움이 안 되고,
`수시/정시/기타`가 일반 전형인 학생은 외국인 특별전형 입학생과 성격이 다릅니다.

**각 필터가 표본을 얼마나 줄이는지 반드시 확인합니다.** 필터 하나가 데이터를 반으로 줄인다면 그걸 모르고 넘어가면 안 됩니다.

**결과를 어떻게 읽나요**
단계별 남은 인원이 출력됩니다. 마지막 줄의 **제적 인원**이 특히 중요합니다.
이 숫자가 변수 개수의 10배보다 적으면 모델이 과적합됩니다.

In [ ]:
def preprocess_undergrad(DF):
    d = DF[DF['학위과정'].isin(['학사과정', '학석사연계과정'])].copy()
    log = [('학위과정: 학사·학석사연계', len(d))]

    d = d[d['입학일자'].notna()]
    log.append(('입학일자 결측 제외', len(d)))

    d['입학연도'] = d['입학연도'].astype(int)
    d = d[d['입학연도'] >= CURRENT_YEAR - YEARS_BACK]
    log.append((f'{CURRENT_YEAR - YEARS_BACK}년 이후 입학', len(d)))

    d = d[d['대학/대학원'] == '대학']
    log.append(('대학 소속', len(d)))

    excluded = d[
        ~d['수시/정시/기타'].str.contains('전기|후기', na=False)
    ]
    print(excluded[['수시/정시/기타']])

    print("---------------------------")


    d = d[d['수시/정시/기타'].str.contains('전기|후기', na=False)]
    log.append(('전기/후기 외국인 전형', len(d)))

    d = add_common_features(d)

    # 신청학점: 결측을 같은 학과 평균으로 채우고, 그래도 남으면 전체 중앙값으로
    d['신청학점'] = pd.to_numeric(d['신청학점'], errors='coerce')
    d['신청학점'] = d['신청학점'].fillna(d.groupby('소속 학과(학부)')['신청학점'].transform('mean'))
    d['신청학점'] = d['신청학점'].fillna(d['신청학점'].median())

    # 학년당 신청학점: 고학년일수록 누적 학점이 많으므로 학년으로 나눠 학업량을 비교 가능하게 만듭니다
    d['학년당신청학점'] = d['신청학점'] / d['학년'].replace(0, np.nan)
    d['학년당신청학점'] = d['학년당신청학점'].fillna(d['학년당신청학점'].median())

    d['학년'] = d['학년'].astype(str)   # 1~4는 크기가 아니라 구분이므로 범주로 다룹니다
    return d, log


def show_filter_log(name, log, d):
    print(f'[{name}]')
    prev = None
    for step, n in log:
        drop = '' if prev is None else f'  (-{prev - n:,})'
        print(f'   {step:28s} {n:6,}{drop}')
        prev = n
    print(f'   {"→ 제적(양성)":28s} {d["제적여부"].sum():6,}  ({d["제적여부"].mean()*100:.1f}%)')


df1, log1 = preprocess_undergrad(DF)
show_filter_log('학부생', log1, df1)


### 4-5. 학습용과 예측용으로 나누기

**무엇을 하나요**
결과가 확정된 학생(졸업·수료·제적)과 아직 진행 중인 학생(재학·휴학)을 나눕니다.

**왜 필요한가요**
재학생은 아직 제적될지 졸업할지 모릅니다. 정답이 없으니 **배울 수 없습니다.**
대신 이들이 바로 우리가 위험을 알고 싶은 대상입니다. 배우는 집단과 예측하는 집단이 다릅니다.

**결과를 어떻게 읽나요**
두 집단의 인원과, 학습용 집단의 제적 인원을 봅니다.
제적 인원이 200명 안팎이면 변수 15개 정도까지가 무리 없는 수준입니다.

In [ ]:

df1

In [ ]:
def split_train_predict(d):
    predict_set = d[d['학적상태'].isin(['재학', '휴학'])].copy()          # 아직 결과를 모르는 학생
    history_set = d[~d['학번'].isin(predict_set['학번'])].copy()          # 결과가 확정된 학생
    return history_set.reset_index(drop=True), predict_set.reset_index(drop=True)


hist1, pred1 = split_train_predict(df1)

summary = pd.DataFrame({
    '학습용(결과 확정)': [len(hist1)],
    '  └ 제적':        [int(hist1['제적여부'].sum())],
    '  └ 제적률':      [f"{hist1['제적여부'].mean()*100:.1f}%", ],
    '예측용(재학·휴학)': [len(pred1)],
}, index=['학부생'])
summary

> **📌 클래스 불균형 (class imbalance)**
>
> 제적 학생이 전체의 15%뿐입니다. 이렇게 한쪽이 훨씬 많은 상태를 불균형이라고 합니다.
> 불균형 데이터에서는 모델이 "전부 다 탈락 안 함"이라고 찍는 쪽으로 쉽게 기울어집니다. 그래도 성적이 좋아 보이니까요.
> 뒤에서 이 문제를 `class_weight`와 임계값 조정으로 다룹니다.

#### ✅ 체크포인트

1. 왜 재학생을 학습에 쓰지 않나요?

<details><summary>답 보기</summary>

1. 재학생은 아직 제적될지 졸업할지 결과가 나오지 않았습니다. 정답이 없는 데이터로는 규칙을 배울 수 없습니다.
   이들은 배움의 대상이 아니라 **적용의 대상**입니다.

</details>

---

## 5. 학부생 모델 만들기

여기서부터는 **학부생 데이터로 한 단계씩** 진행합니다.

### 5-1. 학습셋과 검증셋 나누기

**무엇을 하나요**
학습용 데이터를 다시 7:3으로 쪼개, 3할은 성적표용으로 남겨둡니다.

**왜 필요한가요**
같은 문제로 공부하고 같은 문제로 시험을 보면 점수가 실력을 말해주지 않습니다.
모델이 규칙을 익혔는지, 답을 외웠는지 구분하려면 **한 번도 보지 않은 데이터**가 필요합니다.

**결과를 어떻게 읽나요**
양쪽의 제적 비율이 거의 같은지 확인하세요. 크게 다르면 `stratify`가 빠진 것입니다.

> **📌 층화 추출 (stratified sampling)**
>
> 나눌 때 특정 기준의 비율을 양쪽에서 똑같이 유지하는 방법입니다.
> 제적 학생이 10%뿐인데 운 나쁘게 한쪽에 몰리면 모델이 제대로 배우지 못합니다.
> 반을 두 개로 나눌 때 남녀 비율을 양쪽 다 맞추는 것과 같습니다.

In [ ]:
def make_split(history, num_cols, cat_cols):
    X = history[num_cols + cat_cols]
    y = history['제적여부']
    # X와 y를 함께 넘기므로 짝이 어긋날 수 없습니다
    return train_test_split(X, y, test_size=TEST_SIZE,
                            random_state=RANDOM_STATE, stratify=y)


# 학부생 변수 — 먼저 '소속 대학'까지 넣어보고, 다음 절에서 문제를 확인합니다
NUM_U = ['이수학기수', '최종GPA', '휴학학기수', '학년당신청학점', 'topik']
CAT_U = ['성별', '계열', '국적범주화', '학년', '소속 대학']

X_train_u, X_test_u, y_train_u, y_test_u = make_split(hist1, NUM_U, CAT_U)

print(f'학습셋 {len(X_train_u):,}명 (제적 {y_train_u.sum()}명, {y_train_u.mean()*100:.1f}%)')
print(f'검증셋 {len(X_test_u):,}명 (제적 {y_test_u.sum()}명, {y_test_u.mean()*100:.1f}%)')

### 5-2. 전처리 파이프라인 만들기

**무엇을 하나요**
결측 채우기 → 스케일링 → 더미화 → 모델 학습을 **하나로 묶습니다.**

**왜 필요한가요**
전처리를 파이프라인 밖에서 하면 데이터 누수가 생깁니다.
예를 들어 전체 데이터의 평균으로 스케일링한 뒤 나누면, 학습 단계에서 이미 검증셋의 평균을 엿본 셈입니다.
파이프라인에 넣으면 **교차검증의 매 단계마다 전처리가 그 조각의 데이터만 보고 다시 계산**됩니다.

**결과를 어떻게 읽나요**
파이프라인 구조도가 출력됩니다. 수치 변수와 범주 변수가 각각 다른 경로로 들어가는지 확인하세요.

> **📌 데이터 누수 (data leakage)**
>
> 모델이 원래 알 수 없어야 할 정보를 학습 중에 엿보는 것입니다.
> 시험 문제를 미리 본 것과 같아서, 성적은 잘 나오지만 실전에서는 그만큼 안 나옵니다.

In [ ]:
def build_pipeline(num_cols, cat_cols, C=1.0):
    numeric = Pipeline([
        ('impute', SimpleImputer(strategy='median')),   # 빈칸은 중앙값으로 (평균은 이상치에 흔들립니다)
        ('scale',  StandardScaler()),                   # 평균 0, 표준편차 1로 맞춥니다
    ])
    categorical = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(drop='first',                        # 기준 범주 하나를 뺍니다
                                 sparse_output=False, handle_unknown='ignore')),
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric, num_cols),
        ('cat', categorical, cat_cols),
    ])
    return Pipeline([
        ('pre', preprocessor),
        ('lr',  LogisticRegression(penalty='l2', C=C,
                                   class_weight='balanced',   # 적은 쪽(제적)에 가중치를 줍니다
                                   solver='lbfgs', max_iter=2000,
                                   random_state=42)),
    ])


pipe_u = build_pipeline(NUM_U, CAT_U)
pipe_u

설정 세 가지를 짚고 갑니다.


**`class_weight='balanced'`** — 제적 학생이 10%뿐이라 모델이 "전부 탈락 안 함"으로 기울기 쉽습니다.
이 설정은 적은 쪽의 오답에 더 큰 벌점을 매겨 균형을 맞춥니다.
**대신 예측 확률이 전반적으로 위로 밀립니다.** 그래서 나중에 임계값이 0.5가 아니라 0.8 근처가 나옵니다.
확률값 자체를 "실제 제적 확률"로 읽으면 안 되고, **순위와 임계값**으로 써야 합니다.

**`penalty='l2'`** — 계수가 지나치게 커지지 않도록 잡아주는 안전장치입니다.
어떤 변수가 정답과 거의 완벽히 일치하면 계수가 무한대로 발산하려 하는데, 이걸 막아줍니다.

> **📌 정규화 (regularization)**
>
> 모델이 학습 데이터에 너무 딱 맞추려는 것을 억제하는 장치입니다.
> 시험 범위를 통째로 외우는 대신 원리를 익히게 만드는 것과 비슷합니다.
> `C` 값이 작을수록 더 강하게 억제합니다.

### 5-3. 다중공선성 진단 — 실제로 문제가 터지는 장면

**무엇을 하나요**
변수들끼리 서로 겹치지 않는지 VIF로 확인합니다.

**왜 필요한가요**
회귀는 "다른 조건이 같을 때 이 변수만 1 올리면"을 계산합니다.
그런데 두 변수가 항상 같이 움직이면 "이 변수만 올린" 상황 자체가 데이터에 없습니다.
계수를 정할 근거가 없어지니 값이 제멋대로 튀고, 심하면 계산이 아예 안 됩니다.

**결과를 어떻게 읽나요**
VIF가 `inf`(무한대)로 나오는 변수가 있는지 보세요. 있다면 완전히 겹치는 변수가 있다는 뜻입니다.

> **📌 다중공선성 (multicollinearity)**
>
> 설명 변수들끼리 서로 강하게 겹치는 상태입니다.
> 키와 신발 사이즈를 같이 넣고 "키의 순수한 효과"를 물으면 답할 수 없는 것과 같습니다.
>
> **📌 VIF (분산팽창계수)**
>
> 한 변수를 나머지 변수들로 얼마나 잘 예측할 수 있는지를 나타냅니다.
> 1이면 전혀 안 겹치고, 10을 넘으면 조치가 필요하며, **무한대면 완전히 겹칩니다.**

In [ ]:
def clean_names(preprocessor):
    # ColumnTransformer가 붙이는 'num__', 'cat__' 접두어를 떼고,
    # sklearn이 희소 범주를 묶을 때 쓰는 'infrequent_sklearn'을 읽기 쉬운 말로 바꿉니다
    return [n.split('__', 1)[1].replace('infrequent_sklearn', '기타(소수범주)')
            for n in preprocessor.get_feature_names_out()]


def vif_table(preprocessor, X_train):
    Xt = preprocessor.fit_transform(X_train)
    names = clean_names(preprocessor)
    return pd.DataFrame({
        '변수': names,
        'VIF': [variance_inflation_factor(Xt, i) for i in range(Xt.shape[1])],
    }).sort_values('VIF', ascending=False).reset_index(drop=True)


vif_u = vif_table(build_pipeline(NUM_U, CAT_U).named_steps['pre'], X_train_u)
print(f"VIF가 무한대인 변수: {np.isinf(vif_u['VIF']).sum()}개")
vif_u.round(2)

1.000800e+15는 과학적 표기법으로: 1.000800 × 10^15
즉 대략: 1,000,800,000,000,000

VIF	해석
1 근처	다중공선성 거의 없음



> 1~5	보통 허용 가능
>
> 5 이상	주의 필요
>
> 10 이상	다중공선성이 매우 강함
>
> 1e15 수준	사실상 완전한 선형관계에 가까움
>

In [ ]:
# 소속 대학별로 계열이 몇 종류인지 세어봅니다
cross = pd.crosstab(X_train_u['소속 대학'], X_train_u['계열'])
계열수 = (cross > 0).sum(axis=1)

print(f'소속 대학 {len(계열수)}개 중, 계열이 1개뿐인 대학: {(계열수 == 1).sum()}개')
print()
cross

**원인이 드러났습니다.** 예술대학은 전부 예체능, 문과대학은 전부 인문사회, 사회과학대학도 전부 인문사회입니다.
즉 **`소속 대학`을 알면 `계열`이 자동으로 정해집니다.** `계열`은 `소속 대학` 더미들의 합으로 완벽하게 재현됩니다.


In [ ]:
pre_bad = build_pipeline(NUM_U, CAT_U).named_steps['pre']
Xt_bad = pre_bad.fit_transform(X_train_u)
names_bad = clean_names(pre_bad)

try:
    sm.Logit(y_train_u.values, sm.add_constant(pd.DataFrame(Xt_bad, columns=names_bad))).fit(disp=False)
    print('정상적으로 적합되었습니다')
except np.linalg.LinAlgError as e:
    print('LinAlgError 발생:', e)
    print()
    print('계수를 구하려면 행렬을 역으로 뒤집어야 하는데(역행렬),')
    print('한 열이 다른 열들로 완전히 만들어지면 그 역행렬이 존재하지 않습니다.')
    print('수학적으로 답이 하나로 정해지지 않는다는 뜻입니다.')

**해결: 둘 중 하나를 뺍니다.**

| 선택지 | 장점 | 단점 |
|---|---|---|
| `소속 대학`을 남긴다 | 더 세밀함 | 더미가 여러 개 늘고, 대학별 인원이 적어 불안정 |
| **`계열`을 남긴다** ✅ | 4개 범주로 해석이 쉽고 인원도 충분 | 대학별 차이는 못 봄 |

강의용으로는 **해석이 쉬운 `계열`** 을 남깁니다. 실무에서 대학별 차이가 꼭 필요하다면 반대로 고르면 됩니다.
중요한 건 **둘 다 넣을 수는 없다**는 사실입니다.

> **💡 sklearn은 왜 에러를 내지 않았을까요?**
>
> `LogisticRegression`은 L2 정규화가 걸려 있어 겹치는 변수에도 답을 하나 골라 냅니다. 에러 없이 돌아갑니다.
> 하지만 그 계수는 **여러 가능한 답 중 하나일 뿐**이라 해석하면 안 됩니다.
> statsmodels가 에러를 낸 덕분에 문제를 발견한 것입니다.
> **에러가 나는 도구가 조용히 넘어가는 도구보다 나을 때가 있습니다.**

In [ ]:
# '소속 대학'을 빼고 최종 변수를 확정합니다
CAT_U = ['성별', '계열', '국적범주화', '학년']

X_train_u, X_test_u, y_train_u, y_test_u = make_split(hist1, NUM_U, CAT_U)
pipe_u = build_pipeline(NUM_U, CAT_U)

vif_u = vif_table(build_pipeline(NUM_U, CAT_U).named_steps['pre'], X_train_u)
print(f"VIF 무한대: {np.isinf(vif_u['VIF']).sum()}개  |  VIF > 10: {(vif_u['VIF'] > 10).sum()}개")
vif_u.round(2)

무한대는 사라졌습니다. 다만 `학년_4`의 VIF가 여전히 높습니다(20 안팎). 이건 두 가지 이유입니다.

1. 학부생의 70%가 4학년이라, 이 더미 하나가 데이터 대부분을 설명합니다
2. `학년`과 `이수학기수`는 당연히 함께 움직입니다 (4학년이면 이수학기가 많습니다)

**더미 변수의 높은 VIF는 수치 변수만큼 심각하지 않습니다.** 기준 범주를 무엇으로 잡느냐에 따라 값이 달라지기 때문입니다.
실제로 문제가 되는지는 **계수가 말이 되는지**로 판단합니다. 수치 변수들의 VIF는 모두 3 미만이므로 넘어갑니다.

### 5-4. 모델 학습

**무엇을 하나요**
파이프라인을 학습셋에 맞춥니다. 전처리와 모델이 한 번에 학습됩니다.

**왜 필요한가요**
`fit`을 호출하는 순간 결측 대체값(중앙값), 스케일링 기준(평균·표준편차), 더미 범주 목록이
**학습셋만 보고** 정해집니다. 검증셋에는 이 기준을 그대로 적용만 합니다. 이것이 누수를 막는 구조입니다.

**결과를 어떻게 읽나요**
설계행렬의 크기(행 × 열)를 확인하세요. 열 개수가 더미화 후 실제 변수 개수입니다.

In [ ]:
pipe_u.fit(X_train_u, y_train_u)

pre_u = pipe_u.named_steps['pre']
lr_u  = pipe_u.named_steps['lr']
FEAT_U = clean_names(pre_u)

Xt_train_u = pre_u.transform(X_train_u)
Xt_test_u  = pre_u.transform(X_test_u)

print(f'설계행렬 : {Xt_train_u.shape[0]:,}행 × {Xt_train_u.shape[1]}열')
print(f'제적 사례 : {y_train_u.sum()}명  →  변수 1개당 {y_train_u.sum() / Xt_train_u.shape[1]:.1f}건')
print()
print('만들어진 변수:')
print('  ', ', '.join(FEAT_U))

> **📌 변수 1개당 사례 수 (EPV, events per variable)**
>
> 로지스틱 회귀에서는 **적은 쪽 클래스의 사례 수**를 변수 개수로 나눈 값을 봅니다.
> 보통 10 이상이면 안정적이라고 봅니다. 이 값이 3~4밖에 안 되면 계수가 우연에 크게 흔들립니다.
> 변수를 늘리고 싶을 때마다 이 숫자를 확인하는 습관이 필요합니다.

`계열_인문사회` 같은 이름은 **"인문사회면 1, 아니면 0"** 이라는 뜻입니다.
`계열_기타(소수범주)`는 30명 미만이라 한 덩어리로 묶인 범주들입니다(학부에서는 자연과학).
빠진 `계열_공학`이 기준 범주이므로, `계열_인문사회`의 계수는 **"공학 대비 인문사회의 차이"** 로 읽습니다.

### 5-5. 학년별 제적률 확인 → 학년 제외 → p값 선별 → 최종 회귀

강의에서는 네 단계로 진행합니다. **변수 선택은 학습셋만 사용**하고, 검증셋은 그대로 남겨둡니다.

#### ① 학년별 제적률을 표로 확인하기

제적률(%) = 제적 인원 ÷ 해당 학년 전체 인원 × 100.
이 표는 **재학·휴학을 제외한 학습셋 내부 비율**이며, 학교 전체의 학년별 중도탈락률이 아닙니다.
1~3학년의 비제적이 0명이므로 학년이 제적 여부를 지나치게 잘 구분합니다(준완전분리).
학년은 이 설명용 표에만 남기고, 이후 회귀에서는 제외합니다.

In [ ]:
from IPython.display import display

assert X_train_u.index.equals(y_train_u.index), '학습 데이터와 정답의 행 순서를 확인하세요.'
grade_table = pd.crosstab(X_train_u['학년'], y_train_u).reindex(columns=[0, 1], fill_value=0)
grade_table.columns = ['비제적', '제적']
grade_table['전체'] = grade_table['비제적'] + grade_table['제적']
grade_table['제적률(%)'] = (100 * grade_table['제적'] / grade_table['전체']).round(2)
display(grade_table)

#### ② 학년만 제외하고 오즈비 표 보기

다른 변수는 유지하고 `학년`만 제외하여 같은 학습 학생들로 다시 적합합니다.
`odds_ratio_table`은 계수·표준오차·p값·오즈비·95% 신뢰구간을 반환합니다.

**OR 하한과 OR 상한은 왜 표시하나요?**

오즈비 하나만으로는 추정이 얼마나 불확실한지 알 수 없기 때문입니다.
두 값은 오즈비의 **95% 신뢰구간 양 끝**입니다. 구간이 좁으면 추정이 비교적 정밀하고,
넓으면 불확실성이 큽니다. 오즈비에서 **1은 차이 없음**을 뜻합니다.

| 예시 | 해석 |
|---|---|
| OR 2.0, 하한 1.3, 상한 3.1 | 구간 전체가 1보다 커서 오즈 증가를 뒷받침 |
| OR 2.0, 하한 0.5, 상한 8.0 | 구간이 넓고 1을 포함하므로 증가 방향을 단정하기 어려움 |

따라서 같은 OR 2.0이어도 두 결과의 확실성은 다릅니다. 이 표의 양측 Wald 검정에서는
반올림 경계를 제외하면 95% 구간의 1 포함 여부와 p < .05 여부가 대응합니다.
95%는 이 방식으로 표본추출과 구간 계산을 반복할 때 참값을 포함하는 구간의 비율을 뜻합니다.

수치형 OR은 **학습셋의 1 표준편차 증가**, 범주형 OR은 **기준 범주 대비**입니다.
오즈비는 확률의 비나 인과효과가 아닙니다. 수렴 경고가 나오면 표 해석을 보류하세요.

In [ ]:
def odds_ratio_table(Xt, y, names):
    X_const = sm.add_constant(pd.DataFrame(Xt, columns=names))
    model = sm.Logit(np.asarray(y), X_const).fit(disp=False, maxiter=500)
    ci = np.exp(model.conf_int())  # 계수의 95% 신뢰구간을 OR 단위로 변환
    out = pd.DataFrame({
        '계수': model.params,
        '표준오차': model.bse,
        'p값': model.pvalues,
        '오즈비': np.exp(model.params),
        'OR 하한': ci[0],
        'OR 상한': ci[1],
    })
    out['유의'] = np.where(out['p값'] < .05, '*', '')
    return model, out

def show_odds(table):
    # p값을 0.000으로 반올림하지 않고 작은 값도 표시합니다.
    shown = table.copy()
    for col in ['계수', '표준오차', '오즈비', 'OR 하한', 'OR 상한']:
        shown[col] = shown[col].map(lambda v: f'{v:.3f}')
    shown['p값'] = shown['p값'].map(lambda v: f'{v:.3g}')
    display(shown)

NUM_NO_GRADE = NUM_U.copy()
CAT_NO_GRADE = [c for c in CAT_U if c != '학년']
pre_no_grade = build_pipeline(NUM_NO_GRADE, CAT_NO_GRADE).named_steps['pre']
Xt_no_grade = pre_no_grade.fit_transform(X_train_u[NUM_NO_GRADE + CAT_NO_GRADE])
FEAT_NO_GRADE = clean_names(pre_no_grade)
logit_no_grade, or_no_grade = odds_ratio_table(Xt_no_grade, y_train_u, FEAT_NO_GRADE)
print(f'학년 제외 모형: Pseudo R-squared = {logit_no_grade.prsquared:.4f}')
show_odds(or_no_grade)

#### ③ p값을 참고하여 사용할 변수 고르기

수치형 변수는 위 표의 **p < .05**를 기준으로 한 번만 선별합니다.
범주형은 복잡한 전체 검정 코드를 생략하고 **유지할 원변수 이름을 직접 지정**합니다.
기본 설정은 학년을 제외한 범주형 변수를 모두 유지합니다.

**더미 하나의 p값이 크다는 이유로 범주형 전체를 삭제하거나 특정 더미만 삭제하지 않습니다.**
어떻게 판단해야 하는지는 아래 주석에 남겼습니다. 범주형 유지 목록은 자동 유의성 검정 결과가 아닙니다.

In [ ]:
ALPHA = 0.05

# 수치형: 학년 제외 모형의 p값으로 한 번만 선별합니다.
NUM_KEEP = [c for c in NUM_NO_GRADE if or_no_grade.loc[c, 'p값'] < ALPHA]

# 범주형: 원변수 단위로 직접 선택합니다. 기본값은 학년 외 모두 유지입니다.
# 더미의 p값은 해당 범주와 기준 범주의 차이만 검정합니다.
# 더미 하나가 유의하지 않다고 원변수 전체가 불필요하다는 뜻은 아닙니다.
# 특정 더미만 빼면 그 범주의 효과를 기준 범주와 같다고 가정하게 됩니다.
# 연구 목적과 이론에 따라 유지 여부를 정하세요.
# 통계적으로 전체 변수의 유의성을 판단하려면 관련 더미를 함께 검정해야 합니다.
# 정식 분석에서는 전체 검정(예: 우도비 검정)을 별도로 확인하세요. 여기서는 생략합니다.
# 제외하기로 판단한 원변수는 아래 목록에서 이름을 지우세요.
CAT_KEEP = CAT_NO_GRADE.copy()  # ['성별', '계열', '국적범주화']

KEEP_U = NUM_KEEP + CAT_KEEP
DROP_U = ['학년'] + [c for c in NUM_NO_GRADE + CAT_NO_GRADE if c not in KEEP_U]
selection_table = pd.DataFrame({
    '원변수': NUM_NO_GRADE + CAT_NO_GRADE,
    '선별기준': ['개별 p < .05'] * len(NUM_NO_GRADE) + ['직접 지정'] * len(CAT_NO_GRADE),
    '결정': ['유지' if c in KEEP_U else '제외' for c in NUM_NO_GRADE + CAT_NO_GRADE],
})
display(selection_table)
print('유지:', KEEP_U)
print('제외:', DROP_U)

#### ④ 선택한 변수로 최종 로지스틱 회귀 적합하기

선택된 원변수로 전처리와 회귀를 다시 적합하고 **최종 오즈비 표**를 표시합니다.
절편은 p값에 관계없이 유지합니다. 선택 후 p값은 달라질 수 있습니다.

뒤의 임계값·평가·SHAP도 선택된 변수들을 사용하도록 `pipe_u`, `NUM_U`, `CAT_U`를 갱신합니다.
오즈비는 정규화 없는 statsmodels 모형, 예측·SHAP은 기존 방식의 L2·클래스 가중치 sklearn 모형입니다.
같은 변수를 쓰지만 두 모형의 계수가 같지는 않습니다.

In [ ]:
if not KEEP_U:
    raise ValueError('선택된 변수가 없습니다. p값을 억지로 낮추지 말고 변수 구성과 분석 목적을 검토하세요.')

NUM_U = [c for c in NUM_NO_GRADE if c in KEEP_U]
CAT_U = [c for c in CAT_NO_GRADE if c in KEEP_U]
pipe_u = build_pipeline(NUM_U, CAT_U).fit(X_train_u[KEEP_U], y_train_u)
pre_u = pipe_u.named_steps['pre']
lr_u = pipe_u.named_steps['lr']
FEAT_U = clean_names(pre_u)
Xt_train_u = pre_u.transform(X_train_u[KEEP_U])
Xt_test_u = pre_u.transform(X_test_u[KEEP_U])
logit_u, or_u = odds_ratio_table(Xt_train_u, y_train_u, FEAT_U)
logit_u_selected, or_u_selected = logit_u, or_u

print('최종 사용 변수:', NUM_U + CAT_U)
print(f'최종 모형: Pseudo R-squared = {logit_u.prsquared:.4f}')
print('수렴 여부:', logit_u.mle_retvals['converged'])
show_odds(or_u)

**강의에서 함께 설명할 점**

- p값 선별은 가능한 방법 중 하나이며 필수 절차는 아닙니다. 유의성과 예측력은 다릅니다.
- 선택 후 p값·신뢰구간은 선택 과정의 불확실성을 반영하지 않은 탐색적 결과입니다.
- 뒤의 교차검증은 이미 선택한 변수들을 고정합니다. 변수 선별까지 포함한 전체 절차의 엄밀한 성능 평가는 아니므로, 그 OOF 점수는 낙관적일 수 있습니다.
- 검증셋 정답은 이 변수 선택에 사용하지 않았습니다. 다만 원본의 신청학점 결측 대체가 분할 전 집계에 의존하고, 최종 학적 정보에는 시점 누수가 남아 있습니다. 실무 예측 성능을 보장하지 않습니다.

이제 선택된 변수들로 5-6절의 임계값 분석을 이어갑니다.

### 5-6. 임계값 정하기

**무엇을 하나요**
모델이 내놓은 확률을 몇 점부터 "위험군"으로 볼지 정합니다.

**왜 필요한가요**
모델의 출력은 0과 1 사이 숫자입니다. 0.5를 기준으로 삼는 게 기본값이지만,
불균형 데이터에서 `class_weight='balanced'`를 쓰면 확률이 전체적으로 위로 밀려서 0.5가 맞지 않습니다.

**그리고 검증셋으로 임계값을 고르면 안 됩니다.** 검증셋에 맞춰 고른 임계값으로 그 검증셋 성적을 매기면,
그건 실력이 아니라 "맞춰 놓은 결과"입니다. 그래서 **학습셋 안에서 교차검증**으로 고릅니다.

**결과를 어떻게 읽나요**
선택된 임계값과, 그때의 정밀도·재현율을 봅니다. 그래프에서 두 선이 교차하는 근처가 F1이 최대인 지점입니다.

> **📌 교차검증 (cross-validation)**
>
> 학습셋을 5조각으로 나눠, 4조각으로 배우고 1조각으로 시험보기를 5번 돌아가며 반복합니다.
> 모든 학생이 정확히 한 번씩 "처음 보는 데이터"가 되므로, 검증셋을 건드리지 않고도 공정한 확률을 얻습니다.
> 이렇게 얻은 확률을 **OOF(out-of-fold) 확률**이라고 합니다.

In [ ]:
def find_best_threshold(pipe, X_train, y_train, n_splits=5):
    oof = cross_val_predict(pipe, X_train, y_train,
                            cv=StratifiedKFold(n_splits, shuffle=True, random_state=7),
                            method='predict_proba')[:, 1]
    p, r, th = precision_recall_curve(y_train, oof)
    f1 = 2 * p * r / (p + r + 1e-12)
    best_idx = int(np.nanargmax(f1[:-1]))
    return th[best_idx], oof, (p, r, th, f1)


THRESH_U, oof_u, curve_u = find_best_threshold(pipe_u, X_train_u, y_train_u)
p_u, r_u, th_u, f1_u = curve_u

print(f'선택된 임계값 : {THRESH_U:.4f}')
print(f'그때 정밀도   : {p_u[np.nanargmax(f1_u[:-1])]:.3f}')
print(f'그때 재현율   : {r_u[np.nanargmax(f1_u[:-1])]:.3f}')
print(f'그때 F1       : {np.nanmax(f1_u[:-1]):.3f}')
print()
print(f'참고: 기본값 0.5를 썼다면 위험군으로 분류될 인원 = {(oof_u > 0.5).sum()}명')
print(f'      선택된 임계값을 쓰면                      = {(oof_u > THRESH_U).sum()}명')
print(f'      실제 제적 인원                            = {y_train_u.sum()}명')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(th_u, p_u[:-1], label='정밀도')
axes[0].plot(th_u, r_u[:-1], label='재현율')
axes[0].plot(th_u, f1_u[:-1], label='F1', linestyle='--')
axes[0].axvline(THRESH_U, color='red', linewidth=1, label=f'선택 {THRESH_U:.2f}')
axes[0].set_xlabel('임계값'); axes[0].set_ylabel('점수')
axes[0].set_title('임계값에 따른 성능 변화'); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].hist(oof_u[y_train_u == 0], bins=40, alpha=.6, label='졸업·수료 등')
axes[1].hist(oof_u[y_train_u == 1], bins=40, alpha=.6, label='제적')
axes[1].axvline(THRESH_U, color='red', linewidth=1, label='임계값')
axes[1].set_xlabel('예측 확률'); axes[1].set_ylabel('인원')
axes[1].set_title('예측 확률 분포'); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

왼쪽 그래프에서 **정밀도와 재현율은 서로 반대로 움직입니다.**

> **📌 정밀도(precision)와 재현율(recall)**
>
> - **정밀도**: 위험군이라고 찍은 학생 중 실제로 제적된 비율 → **헛짚지 않았는가**
> - **재현율**: 실제 제적한 학생 중 위험군으로 찍어낸 비율 → **놓치지 않았는가**
>
> 임계값을 낮추면 더 많이 찍으니 놓치는 학생은 줄지만(재현율↑) 헛걸음이 늘어납니다(정밀도↓).
> 건강검진 재검 기준선을 낮추는 것과 같습니다.
>
> **F1**은 둘의 조화평균으로, 어느 한쪽으로 치우치지 않은 지점을 찾아줍니다.

**중도탈락 예측에서는 재현율을 더 중시하는 것이 보통입니다.**
위험 학생을 놓치면 실제로 학생이 학교를 떠나지만, 헛짚어도 상담 한 번 더 하는 것에 그치기 때문입니다.
정책적으로 재현율을 높이고 싶다면 임계값을 F1 최적점보다 낮추면 됩니다.

### 5-7. 성능 평가

**무엇을 하나요**
한 번도 쓰지 않은 검증셋으로 성적을 매깁니다.

**왜 필요한가요**
학습셋 성적은 실력이 아닙니다. 지금까지 이 검증셋은 전처리 기준을 정할 때도, 임계값을 고를 때도 쓰지 않았습니다.
그래서 여기서 나온 숫자가 **실전에 가장 가까운 추정치**입니다.

**결과를 어떻게 읽나요**
정확도가 아니라 **정밀도·재현율·PR-AUC**를 보세요. 함께 출력되는 "전원 음성 예측 시 정확도"와 비교하면
왜 정확도를 보면 안 되는지 알 수 있습니다.

In [ ]:
def evaluate(y_true, prob, threshold, label=''):
    pred = (prob > threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    tn, fp, fn, tp = cm.ravel()

    baseline_acc = max((y_true == 0).mean(), (y_true == 1).mean())

    print(f'=== {label} (임계값 {threshold:.4f}) ===')
    print()
    print('              예측:유지   예측:탈락')
    print(f'  실제:유지   {tn:>8,}   {fp:>8,}')
    print(f'  실제:탈락   {fn:>8,}   {tp:>8,}')
    print()
    print(f'  정밀도  {precision_score(y_true, pred, zero_division=0):.3f}   '
          f'(위험군 {tp+fp}명 중 {tp}명 적중)')
    print(f'  재현율  {recall_score(y_true, pred, zero_division=0):.3f}   '
          f'(실제 탈락 {tp+fn}명 중 {tp}명 포착, {fn}명 놓침)')
    print(f'  F1      {f1_score(y_true, pred, zero_division=0):.3f}')
    print(f'  PR-AUC  {average_precision_score(y_true, prob):.3f}   '
          f'(아무 정보 없을 때 기준선 = {y_true.mean():.3f})')
    print(f'  ROC-AUC {roc_auc_score(y_true, prob):.3f}')
    print()
    print(f'  정확도  {accuracy_score(y_true, pred):.3f}')
    print(f'  └ 전원 "탈락 안 함"으로 찍기만 해도 정확도 {baseline_acc:.3f} 입니다.')
    print(f'    그 모델은 탈락 학생을 한 명도 못 찾습니다. 정확도는 이 문제에 쓸 수 없습니다.')
    return pd.Series({'정밀도': precision_score(y_true, pred, zero_division=0),
                      '재현율': recall_score(y_true, pred, zero_division=0),
                      'F1': f1_score(y_true, pred, zero_division=0),
                      'PR-AUC': average_precision_score(y_true, prob),
                      'ROC-AUC': roc_auc_score(y_true, prob)})


prob_test_u = pipe_u.predict_proba(X_test_u)[:, 1]
score_u = evaluate(y_test_u, prob_test_u, THRESH_U, '학부생 검증셋')

> **📌 PR-AUC vs ROC-AUC**
>
> 둘 다 임계값과 무관하게 모델의 순위 매기는 능력을 봅니다. 하지만 불균형 데이터에서는 다릅니다.
>
> **ROC-AUC는 음성이 압도적으로 많을 때 낙관적으로 보입니다.** 음성을 잘 맞히는 것만으로도 점수가 올라가기 때문입니다.
> **PR-AUC는 소수 클래스(제적)에 집중합니다.** 아무 정보 없이 찍을 때의 기준선이 곧 제적률(약 0.10)이므로,
> PR-AUC가 0.10이면 무작위와 같고 0.85라면 확실히 잘하는 것입니다.
>
> 불균형 문제에서는 **PR-AUC를 주 지표로** 삼으세요.

#### ✅ 체크포인트

1. 임계값을 검증셋이 아니라 학습셋 교차검증으로 고른 이유는?
2. ROC-AUC가 0.96인데도 마냥 기뻐하면 안 되는 이유는 무엇일까요?

<details><summary>답 보기</summary>

1. 검증셋에 맞춰 임계값을 고르면 그 검증셋 성적은 "맞춰 놓은 결과"가 됩니다.
   처음 보는 데이터에서는 그만큼 안 나옵니다. 검증셋은 끝까지 손대지 않아야 성적표로서 의미가 있습니다.
2. 점수가 지나치게 높으면 **누수**를 의심해야 합니다.
   이 데이터의 `이수학기수`·`학년`은 제적자에게는 제적 시점 값이라, 모델이 "결과를 보고 원인을 맞히는" 상황에 가깝습니다.
   8장에서 자세히 다룹니다.

</details>

---

## 6. SHAP — 모델이 무엇을 보고 판단했나

### 6-1. SHAP이란

오즈비는 "이 변수가 1 늘면 오즈가 몇 배"를 알려줍니다. 하지만 현장에서 필요한 건 다른 질문입니다.

> "이 학생은 **왜** 위험군인가요?"

SHAP은 **예측값 하나를 변수별 기여도의 합으로 쪼갭니다.**

```
이 학생의 예측 로그오즈 = 기준값 + (GPA가 기여한 몫) + (휴학학기수가 기여한 몫) + ...
```

시험 총점을 과목별 점수로 쪼개는 것과 같습니다. 총점만 보면 "잘했다/못했다"뿐이지만,
과목별로 쪼개면 **어디를 보강해야 하는지**가 보입니다.

> **📌 SHAP (SHapley Additive exPlanations)**
>
> 게임 이론에서 "여러 명이 함께 만든 성과를 각자의 기여로 공정하게 나누는 방법"을 가져온 것입니다.
> 각 변수를 팀원이라고 보고, 예측값이라는 성과에 누가 얼마나 기여했는지를 계산합니다.

### 6-2. SHAP 값 계산

**무엇을 하나요**
선형 모델 전용 explainer로 검증셋 학생들의 SHAP 값을 구합니다.

**왜 이렇게 하나요**
SHAP은 **전처리가 끝난 설계행렬**에 적용해야 합니다. 파이프라인 전체에 씌우면
원본 컬럼 개수(10개)와 더미화 후 개수(15개)가 맞지 않아 실패합니다.
변수 이름도 반드시 `get_feature_names_out()`에서 가져옵니다.

**결과를 어떻게 읽나요**
SHAP 값 행렬의 크기가 (검증셋 인원 × 변수 개수)와 맞는지 확인합니다.

In [ ]:
explainer_u = shap.LinearExplainer(
    lr_u,
    masker=shap.maskers.Independent(Xt_train_u, max_samples=Xt_train_u.shape[0]))

shap_u = explainer_u(Xt_test_u)
shap_u.feature_names = FEAT_U

print(f'SHAP 값 행렬 : {shap_u.values.shape[0]:,}명 × {shap_u.values.shape[1]}개 변수')
print(f'기준값(base) : {np.ravel(shap_u.base_values)[0]:.4f}')
print()
print('기준값이란 "아무 정보 없이 평균적인 학생이라면" 나올 로그오즈입니다.')
print('여기서 각 변수의 기여가 더해지거나 빠져서 최종 예측이 나옵니다.')

### 6-3. SHAP은 마법이 아닙니다 — 직접 계산해 확인하기

**무엇을 하나요**
선형 모델에서는 SHAP 값을 손으로 계산할 수 있습니다. 두 결과가 같은지 확인합니다.

**왜 필요한가요**
SHAP을 "설명해주는 블랙박스"로 받아들이면, 나중에 결과가 이상해도 의심하지 못합니다.
선형 모델에서 SHAP 값의 정체는 놀랍도록 단순합니다.

```
SHAP값 = 계수 × (그 학생의 값 − 전체 평균)
```

즉 **"평균적인 학생과 비교해서 이 변수가 얼마나 다른가 × 그 변수의 중요도"** 입니다.
평균과 같으면 기여가 0이고, 평균에서 멀수록 기여가 커집니다.

**결과를 어떻게 읽나요**
`True`가 나오면 두 계산이 완전히 일치한 것입니다.

In [ ]:
# 손으로 계산: 계수 × (값 - 학습셋 평균)
manual = lr_u.coef_[0] * (Xt_test_u - Xt_train_u.mean(axis=0))

print('SHAP 라이브러리 결과와 수동 계산이 일치하는가 :',
      np.allclose(manual, shap_u.values, atol=1e-8))
print(f'최대 오차 : {np.abs(manual - shap_u.values).max():.2e}')
print()

# 가법성: 기준값 + 모든 SHAP 값의 합 = 모델의 실제 출력
총합 = np.ravel(shap_u.base_values) + shap_u.values.sum(axis=1)
print('기준값 + SHAP 합계 = 모델 출력(로그오즈) 인가 :',
      np.allclose(총합, lr_u.decision_function(Xt_test_u), atol=1e-8))
print()

# 학생 한 명으로 눈으로 확인
i = 0
표 = pd.DataFrame({
    '변수': FEAT_U,
    '계수': lr_u.coef_[0].round(3),
    '이 학생 값(표준화)': Xt_test_u[i].round(3),
    '평균': Xt_train_u.mean(axis=0).round(3),
    'SHAP': shap_u.values[i].round(3),
})
print(f'검증셋 {i}번 학생의 기여도 분해')
표

> **📌 `max_samples`를 지정한 이유**
>
> `shap.maskers.Independent`는 기본적으로 배경 데이터를 100행만 표본추출해서 평균을 냅니다.
> 그러면 그 표본 평균이 전체 평균과 미세하게 달라 수동 계산과 정확히 맞지 않습니다.
> 위에서는 등식을 눈으로 확인하려고 학습셋 전체를 배경으로 지정했습니다.

### 6-4. 전역 중요도 — 전체적으로 어떤 요인이 중요한가

**무엇을 하나요**
모든 학생의 SHAP 값을 모아 변수별로 요약합니다. 두 가지 그림을 그립니다.

**왜 두 가지인가요**
- **막대 그래프**는 "얼마나 중요한가"만 보여줍니다 (크기 순서)
- **beeswarm**은 "어느 방향으로 미는가"까지 보여줍니다 (값이 높을 때 위험이 오르는지 내리는지)

중요도만으로는 정책을 세울 수 없습니다. **방향을 알아야 개입할 수 있습니다.**

**결과를 어떻게 읽나요**
beeswarm에서 점 하나가 학생 한 명입니다.
- 가로 위치: 오른쪽일수록 탈락 위험을 **올리는** 기여
- 색: 빨강 = 그 변수의 값이 큼 / 파랑 = 값이 작음

예를 들어 `최종GPA` 줄에서 **파란 점이 오른쪽**에 몰려 있다면, "GPA가 낮은 학생이 위험이 올라간다"는 뜻입니다.

In [ ]:
imp_u = (pd.DataFrame({'변수': FEAT_U, '평균 |SHAP|': np.abs(shap_u.values).mean(axis=0)})
         .sort_values('평균 |SHAP|', ascending=False).reset_index(drop=True))
imp_u.round(4)

In [ ]:
plt.figure()
shap.plots.bar(shap_u, max_display=15, show=False)
plt.title('변수별 평균 기여도 (학부생)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure()
shap.plots.beeswarm(shap_u, max_display=15, show=False)
plt.title('SHAP 분포 — 오른쪽일수록 탈락 위험을 올림 (학부생)', fontsize=12)
plt.tight_layout(); plt.show()

### 6-5. 개별 학생 설명 — 상담에 바로 쓰는 그림

**무엇을 하나요**
가장 위험도가 높게 나온 학생 한 명의 예측을 변수별로 분해합니다.

**왜 필요한가요**
전역 중요도 그림 열 장보다 **이 그림 한 장이 현장을 설득합니다.**
"왜 이 학생을 부르라는 거죠?"라는 질문에 답할 수 있는 건 이 그림뿐입니다.

**결과를 어떻게 읽나요**
맨 아래 `E[f(x)]`가 평균적인 학생의 출발점이고, 맨 위 `f(x)`가 이 학생의 최종 점수입니다.
빨간 막대는 위험을 **올린** 요인, 파란 막대는 **내린** 요인입니다. 막대 길이가 기여 크기입니다.

In [ ]:
위험순위 = np.argsort(-prob_test_u)
i_top = int(위험순위[0])

print(f'검증셋에서 위험도가 가장 높은 학생 (예측 확률 {prob_test_u[i_top]:.3f})')
print(f'실제 결과: {"제적" if y_test_u.iloc[i_top] == 1 else "제적 아님"}')
print()
X_test_u.iloc[[i_top]]

In [ ]:
plt.figure()
shap.plots.waterfall(shap_u[i_top], max_display=12, show=False)
plt.title('위험도 1위 학생의 예측 분해', fontsize=12)
plt.tight_layout(); plt.show()

### 6-6. 주요 변수 자세히 보기

**무엇을 하나요**
가장 중요한 변수 두 개에 대해, 변수 값과 기여도의 관계를 흩뿌려 그립니다.

**왜 필요한가요**
"GPA가 중요하다"만으로는 부족합니다. **GPA가 몇 점 아래로 떨어질 때 위험이 급해지는지**를 알아야
"평점 2.0 미만 학생 전수 상담" 같은 실행 가능한 기준을 만들 수 있습니다.

**결과를 어떻게 읽나요**
가로축이 변수 값(표준화된 값), 세로축이 기여도입니다.
로지스틱 회귀는 직선만 그릴 수 있으므로 **점들이 직선 위에 놓입니다.**
XGBoost 노트북의 같은 그림과 비교하면 차이가 뚜렷하게 보입니다.

In [ ]:
top2 = imp_u['변수'].head(2).tolist()
for name in top2:
    plt.figure()
    shap.plots.scatter(shap_u[:, name], color=shap_u, show=False)
    plt.title(f'{name}의 값과 기여도', fontsize=12)
    plt.tight_layout(); plt.show()

**점이 정확히 직선 위에 놓입니다.** 이것이 로지스틱 회귀의 가장 큰 한계입니다.

실제로는 "GPA 2.0 아래에서 위험이 급격히 커지다가 3.0 위에서는 별 차이 없음" 같은 꺾인 관계일 수 있는데,
로지스틱 회귀는 그걸 표현할 수 없습니다. 직선 하나로만 설명하기 때문입니다.

**이걸 다루는 방법은 두 가지입니다.**
1. 변수를 구간으로 나눠서 더미로 만든다 (예: GPA 0~2, 2~3, 3~4.5)
2. 비선형을 자동으로 학습하는 모델을 쓴다 → **XGBoost 노트북에서 확인하세요**

> **📌 선형 가정 (linearity assumption)**
>
> 로지스틱 회귀는 "설명 변수가 1 늘면 로그오즈가 항상 일정하게 변한다"고 가정합니다.
> 현실이 그렇지 않을 때 이 가정은 모델의 한계가 됩니다. 대신 **해석이 단순명료하다**는 장점을 얻습니다.

### 6-7. 오즈비와 SHAP은 왜 순위가 다를까

**무엇을 하나요**
두 가지 중요도 순위를 나란히 놓고 비교합니다.

**왜 필요한가요**
둘이 다르다고 해서 하나가 틀린 게 아닙니다. **서로 다른 질문에 답하고 있습니다.**

| | 질문 | 특징 |
|---|---|---|
| **오즈비 (계수)** | 다른 조건이 같을 때, 이 변수가 1 늘면? | 변수가 실제로 얼마나 변하는지는 반영하지 않음 |
| **SHAP** | 실제 데이터에서 이 변수가 평균적으로 얼마나 기여했나? | 변수의 분포를 반영 |

거의 변하지 않는 변수는 계수가 커도 실제 기여는 작습니다.
반대로 계수가 작아도 학생마다 값이 크게 다르면 기여가 클 수 있습니다.

**결과를 어떻게 읽나요**
`순위차이`가 큰 변수를 찾아보세요. 그 변수가 바로 "계수는 크지만 실제로는 잘 안 변하는" 변수입니다.

In [ ]:
비교 = (or_u.drop(index='const', errors='ignore')
        .assign(계수크기=lambda d: d['계수'].abs())
        .reset_index().rename(columns={'index': '변수'})[['변수', '계수', '오즈비', 'p값', '계수크기']])
비교 = 비교.merge(imp_u, on='변수', how='outer')
비교['계수 순위'] = 비교['계수크기'].rank(ascending=False)
비교['SHAP 순위'] = 비교['평균 |SHAP|'].rank(ascending=False)
비교['순위차이'] = (비교['계수 순위'] - 비교['SHAP 순위']).abs()
비교.sort_values('SHAP 순위')[['변수', '오즈비', 'p값', '평균 |SHAP|', '계수 순위', 'SHAP 순위', '순위차이']].round(3)

### 6-8. 위험군 명단 만들기

**무엇을 하나요**
재학·휴학 중인 학생에게 모델을 적용해 위험군 명단을 뽑고, 각 학생마다 **위험 요인 3가지**를 붙입니다.

**왜 필요한가요**
확률 숫자만 있는 명단은 현장에서 쓰기 어렵습니다.
"이 학생은 휴학이 길고 성적이 낮아서 위험합니다"까지 있어야 상담으로 이어집니다.

**결과를 어떻게 읽나요**
위험순위 1번부터 확인하세요. `주요위험요인`은 **"변수 상태 (방향 기여도)"** 형식입니다.
예를 들어 `최종GPA 낮음 (위험↑ 5.60)`은 "GPA가 평균보다 낮아서 위험을 5.60만큼 올렸다"는 뜻입니다.

In [ ]:
def build_risk_list(pipeline, explainer_bg_Xt, predict_df, num_cols, cat_cols,
                    threshold, feat_names, top_k=3):
    n_num = len(num_cols)   # 앞쪽 n_num개가 수치 변수, 나머지는 0/1 더미입니다
    pre = pipeline.named_steps['pre']
    lr  = pipeline.named_steps['lr']

    X_pred = predict_df[num_cols + cat_cols]
    prob   = pipeline.predict_proba(X_pred)[:, 1]

    Xt_pred = pre.transform(X_pred)
    expl = shap.LinearExplainer(
        lr, masker=shap.maskers.Independent(explainer_bg_Xt,
                                            max_samples=explainer_bg_Xt.shape[0]))
    sv = expl(Xt_pred)

    def describe(j, shap_val, x_val):
        # 수치 변수는 표준화되어 있으므로 부호가 곧 평균 대비 높낮이입니다
        if j < n_num:
            상태 = '높음' if x_val > 0 else '낮음'
        else:
            상태 = '해당' if x_val > 0.5 else '비해당'
        방향 = '위험↑' if shap_val > 0 else '위험↓'
        return f'{feat_names[j]} {상태} ({방향} {abs(shap_val):.2f})'

    def top_factors(shap_row, x_row):
        idx = np.argsort(-np.abs(shap_row))[:top_k]
        return ' | '.join(describe(j, shap_row[j], x_row[j]) for j in idx)

    out = predict_df.copy()
    out['위험확률'] = prob.round(4)
    out['위험군']   = (prob > threshold).astype(int)
    out['주요위험요인'] = [top_factors(sv.values[k], Xt_pred[k]) for k in range(len(out))]

    risk = (out[out['위험군'] == 1]
            .sort_values('위험확률', ascending=False)
            .reset_index(drop=True))
    risk.insert(0, '위험순위', range(1, len(risk) + 1))
    return out, risk, sv


COLS_OUT = ['위험순위', '학번', '성명', '소속 대학', '소속 학과(학부)', '소속 전공',
            '학위과정', '학적상태', '국적', '학년', '최종GPA', '휴학학기수',
            '위험확률', '주요위험요인']

pred1_scored, risk1, shap_pred1 = build_risk_list(
    pipe_u, Xt_train_u, pred1, NUM_U, CAT_U, THRESH_U, FEAT_U)

print(f'예측 대상 {len(pred1):,}명 중 위험군 {len(risk1):,}명 ({len(risk1)/len(pred1)*100:.1f}%)')
risk1[[c for c in COLS_OUT if c in risk1.columns]].head(15)

In [ ]:
out_path_u = '[명단]외국인학부생_중도탈락위험군_로지스틱회귀.csv'
risk1[[c for c in COLS_OUT if c in risk1.columns]].to_csv(out_path_u, encoding='ms949', index=False)
print(f'저장 완료: {out_path_u}  ({len(risk1):,}행)')

> **💡 `encoding='ms949'` 로 저장하는 이유**
>
> 윈도우 엑셀은 CSV를 열 때 기본적으로 `ms949`(한국어 윈도우 인코딩)로 읽습니다.
> `utf-8`로 저장하면 담당 선생님이 열었을 때 한글이 전부 깨집니다.
> **결과물은 받는 사람의 환경에 맞춰야 합니다.**

### 6-9. 전체 요인 인사이트

**무엇을 하나요**
지금까지의 결과를 문장으로 정리합니다.

**왜 필요한가요**
보고서에 그대로 옮길 수 있는 형태가 필요합니다. 그림만 붙이면 읽는 사람이 해석해야 합니다.

In [ ]:
def make_insight(imp, or_table, feat_names, shap_values, label):
    lines = [f'── {label} 중도탈락 요인 분석 ──', '']
    for rank, (_, row) in enumerate(imp.head(5).iterrows(), 1):
        name = row['변수']
        j = feat_names.index(name)
        # 그 변수의 값이 클 때 SHAP이 커지는지 작아지는지로 방향을 봅니다
        col = shap_values.data[:, j]
        corr = np.corrcoef(col, shap_values.values[:, j])[0, 1] if np.std(col) > 0 else 0
        방향 = '값이 높을수록 위험 ↑' if corr > 0 else '값이 높을수록 위험 ↓'
        오즈비 = or_table['오즈비'].get(name, np.nan)
        유의 = or_table['유의'].get(name, '')
        lines.append(f'{rank}. {name}')
        lines.append(f'   평균 기여도 {row["평균 |SHAP|"]:.3f}  |  {방향}'
                     + (f'  |  오즈비 {오즈비:.2f} {유의}' if not np.isnan(오즈비) else ''))
    lines.append('')
    lines.append('* 오즈비는 "다른 조건이 같을 때의 효과", 기여도는 "실제 분포에서의 평균 영향"입니다.')
    lines.append('* ***: p<0.001, **: p<0.01, *: p<0.05')
    return '\n'.join(lines)


print(make_insight(imp_u, or_u, FEAT_U, shap_u, '학부생'))

---

## 7. 대학원생 모델

같은 절차를 대학원생에게 적용합니다. **절차는 같고 변수만 다릅니다.**

| | 학부생 | 대학원생 |
|---|---|---|
| 수치 변수 | 이수학기수, 최종GPA, 입학시나이, 휴학학기수, **학년당**신청학점, topik | 이수학기수, 최종GPA, 입학시나이, 휴학학기수, **학기당**신청학점 |
| 범주 변수 | 성별, 계열, 국적범주화, **학년** | 성별, 계열, 국적범주화, **학위과정** |

`topik`을 대학원에서 빼는 이유는 대학원 입학은 TOPIK 외에 영어 성적으로도 가능해
결측이 "미제출"인지 "다른 요건으로 입학"인지 구분되지 않기 때문입니다.

`대학원구분`(일반/전문/특수)도 뺍니다. 다음 셀에서 그 이유를 확인합니다.

**실행 안내**: 첨부 원본에는 대학원 전처리와 `hist2`, `pred2` 정의가 없습니다.
따라서 이 사본은 필요한 데이터·열이 준비된 경우에만 아래 대학원 절을 실행합니다.
이번에 추가한 진단·변수 선별·재적합 대상은 앞에서 실제 정의된 **학부생 모형**입니다.

In [ ]:
_graduate_cols = ['이수학기수', '최종GPA', '입학시나이', '휴학학기수', '학기당신청학점',
                 '성별', '계열', '국적범주화', '학위과정', '대학원구분', '제적여부']
_graduate_ready = (all(k in globals() for k in ['hist2', 'pred2'])
                   and all(c in hist2.columns for c in _graduate_cols)
                   and all(c in pred2.columns for c in _graduate_cols))
if not _graduate_ready:
    print('대학원 절 건너뜀: 원본에 대학원 전처리 데이터/필수 열이 정의되지 않았습니다.')

### 7-1. 대학원에서도 공선성 확인

**무엇을 하나요**
학부에서 `소속 대학`과 `계열`이 겹쳤던 것처럼, 대학원에서도 겹치는 변수가 있는지 봅니다.

**왜 필요한가요**
집단이 바뀌면 변수 간 관계도 바뀝니다. **학부에서 괜찮았다고 대학원에서도 괜찮은 게 아닙니다.**
매번 확인해야 합니다.

**결과를 어떻게 읽나요**
교차표에서 한 줄에 값이 한 칸에만 몰려 있으면 그 두 변수는 겹칩니다.

In [ ]:
if _graduate_ready:
    교차 = pd.crosstab(hist2['대학원구분'], hist2['계열'])
    print('대학원구분 × 계열')
    print(교차.to_string())
    print()
    계열종류 = (교차 > 0).sum(axis=1)
    for 구분, n in 계열종류.items():
        상태 = '계열 1개로 고정 → 겹침' if n == 1 else f'계열 {n}종'
        print(f'  {구분:4s} : {상태}')

`전문`대학원은 전원 예체능(영상대학원), `특수`는 전원 인문사회입니다.
학부 때와 같은 구조의 겹침이라 **`대학원구분`을 빼고 `계열`을 남깁니다.**

In [ ]:
if _graduate_ready:
    NUM_G = ['이수학기수', '최종GPA', '입학시나이', '휴학학기수', '학기당신청학점']
    CAT_G = ['성별', '계열', '국적범주화', '학위과정']

    X_train_g, X_test_g, y_train_g, y_test_g = make_split(hist2, NUM_G, CAT_G)
    pipe_g = build_pipeline(NUM_G, CAT_G).fit(X_train_g, y_train_g)

    pre_g = pipe_g.named_steps['pre']
    lr_g  = pipe_g.named_steps['lr']
    FEAT_G = clean_names(pre_g)
    Xt_train_g = pre_g.transform(X_train_g)
    Xt_test_g  = pre_g.transform(X_test_g)

    print(f'학습셋 {len(X_train_g):,}명 (제적 {y_train_g.sum()}명, {y_train_g.mean()*100:.1f}%)')
    print(f'검증셋 {len(X_test_g):,}명 (제적 {y_test_g.sum()}명)')
    print(f'설계행렬 {Xt_train_g.shape[0]:,} × {Xt_train_g.shape[1]}  |  '
          f'변수 1개당 제적 사례 {y_train_g.sum()/Xt_train_g.shape[1]:.1f}건')
    print()
    vif_g = vif_table(build_pipeline(NUM_G, CAT_G).named_steps['pre'], X_train_g)
    print(f"VIF 무한대 {np.isinf(vif_g['VIF']).sum()}개  |  VIF>10 {(vif_g['VIF']>10).sum()}개  |  "
          f"최대 {vif_g['VIF'].max():.1f} ({vif_g.iloc[0]['변수']})")

### 7-2. 계수 해석과 성능

**무엇을 하나요**
오즈비 표와 검증셋 성능을 한 번에 봅니다.

**결과를 어떻게 읽나요**
학부와 비교해 보세요. 제적률이 절반 수준(약 4.6%)이라 **양성 사례가 적고, 그만큼 재현율이 낮게 나옵니다.**
드문 사건일수록 예측이 어렵습니다.

In [ ]:
if _graduate_ready:
    logit_g, or_g = odds_ratio_table(Xt_train_g, y_train_g, FEAT_G)
    print(f'Pseudo R-squared : {logit_g.prsquared:.4f}')
    or_g.reindex(or_g['계수'].abs().sort_values(ascending=False).index).round(3)

In [ ]:
if _graduate_ready:
    THRESH_G, oof_g, curve_g = find_best_threshold(pipe_g, X_train_g, y_train_g)
    prob_test_g = pipe_g.predict_proba(X_test_g)[:, 1]
    score_g = evaluate(y_test_g, prob_test_g, THRESH_G, '대학원생 검증셋')

### 7-3. 대학원생 SHAP 분석

In [ ]:
if _graduate_ready:
    explainer_g = shap.LinearExplainer(
        lr_g, masker=shap.maskers.Independent(Xt_train_g, max_samples=Xt_train_g.shape[0]))
    shap_g = explainer_g(Xt_test_g)
    shap_g.feature_names = FEAT_G

    imp_g = (pd.DataFrame({'변수': FEAT_G, '평균 |SHAP|': np.abs(shap_g.values).mean(axis=0)})
             .sort_values('평균 |SHAP|', ascending=False).reset_index(drop=True))
    imp_g.round(4)

In [ ]:
if _graduate_ready:
    plt.figure()
    shap.plots.beeswarm(shap_g, max_display=15, show=False)
    plt.title('SHAP 분포 — 오른쪽일수록 탈락 위험을 올림 (대학원생)', fontsize=12)
    plt.tight_layout(); plt.show()

In [ ]:
if _graduate_ready:
    i_top_g = int(np.argmax(prob_test_g))
    plt.figure()
    shap.plots.waterfall(shap_g[i_top_g], max_display=12, show=False)
    plt.title('위험도 1위 대학원생의 예측 분해', fontsize=12)
    plt.tight_layout(); plt.show()

In [ ]:
if _graduate_ready:
    pred2_scored, risk2, shap_pred2 = build_risk_list(
        pipe_g, Xt_train_g, pred2, NUM_G, CAT_G, THRESH_G, FEAT_G)

    out_path_g = '[명단]외국인대학원생_중도탈락위험군_로지스틱회귀.csv'
    cols_g = [c for c in COLS_OUT if c in risk2.columns]
    risk2[cols_g].to_csv(out_path_g, encoding='ms949', index=False)

    print(f'예측 대상 {len(pred2):,}명 중 위험군 {len(risk2):,}명 ({len(risk2)/len(pred2)*100:.1f}%)')
    print(f'저장 완료: {out_path_g}')
    risk2[cols_g].head(15)

In [ ]:
if _graduate_ready:
    print(make_insight(imp_g, or_g, FEAT_G, shap_g, '대학원생'))

### 7-4. 두 집단 비교

**무엇을 하나요**
학부생과 대학원생의 결과를 나란히 놓습니다.

**왜 필요한가요**
같은 학교의 유학생이라도 집단마다 위험 요인이 다릅니다.
**한 가지 정책을 모두에게 적용하면 안 된다**는 근거가 여기서 나옵니다.

In [ ]:
if _graduate_ready:
    비교표 = pd.DataFrame({'학부생': score_u, '대학원생': score_g}).round(3)
    비교표.loc['임계값'] = [round(THRESH_U, 3), round(THRESH_G, 3)]
    비교표.loc['제적률(학습)'] = [round(y_train_u.mean(), 3), round(y_train_g.mean(), 3)]
    비교표.loc['위험군 인원'] = [len(risk1), len(risk2)]
    비교표.loc['예측 대상'] = [len(pred1), len(pred2)]
    print('=== 성능 비교 ===')
    print(비교표.to_string())
    print()
    print('=== 주요 요인 Top5 ===')
    print(pd.DataFrame({'학부생': imp_u['변수'].head(5).values,
                        '대학원생': imp_g['변수'].head(5).values},
                       index=[f'{i}위' for i in range(1, 6)]).to_string())

---

## 8. 이 모델의 한계 — 반드시 알고 써야 할 것

성능 지표가 높게 나왔다고 좋아하기 전에, 이 결과를 믿을 수 있는지 따져봐야 합니다.

### 8-1. 시점 누수 — 가장 중요한 한계

**무엇이 문제인가요**

`이수학기수`, `최종GPA`, `휴학학기수`, `학년`은 **학생마다 관측된 시점이 다릅니다.**

| 학생 유형 | 이 값들은 언제 시점의 값인가 |
|---|---|
| 제적한 학생 | **제적한 그 시점** |
| 졸업한 학생 | **졸업한 시점** (8학기 전부 이수) |
| 지금 재학 중인 학생 | **현재** (아직 진행 중) |

3학기 만에 제적한 학생은 `이수학기수 = 3`이고, 졸업생은 `이수학기수 = 8`입니다.
모델은 이걸 보고 "이수학기수가 적으면 제적"이라고 배웁니다.
**하지만 이건 원인이 아니라 결과입니다.** 제적했기 때문에 이수학기가 적은 것이지, 그 반대가 아닙니다.

**결과를 어떻게 읽나요**
아래에서 제적자와 졸업생의 값 분포를 비교하면 이 문제가 눈에 보입니다.

In [ ]:
비교대상 = ['이수학기수', '최종GPA', '휴학학기수']
표 = hist1.groupby(hist1['제적여부'].map({0: '제적 아님', 1: '제적'}))[비교대상].mean().round(2)
print('학부생: 집단별 평균')
print(표.to_string())
print()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, 비교대상):
    for v, lab in [(0, '제적 아님'), (1, '제적')]:
        ax.hist(hist1.loc[hist1['제적여부'] == v, col].dropna(),
                bins=20, alpha=.55, label=lab, density=True)
    ax.set_title(col); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.suptitle('제적 여부에 따른 분포 차이 — 시점이 달라서 생긴 차이입니다', fontsize=11)
plt.tight_layout(); plt.show()

두 분포가 크게 어긋나 있습니다. 모델이 쉽게 맞히는 이유가 여기 있습니다.
**ROC-AUC가 0.95를 넘는 것은 모델이 훌륭해서가 아니라, 문제가 쉬워졌기 때문입니다.**

**그럼 이 모델은 쓸모없나요? 아닙니다.** 다만 이렇게 이해해야 합니다.

- ❌ "이 학생은 탈락할 것이다" — 이렇게 단정하기엔 근거가 약합니다
- ✅ "이 학생은 **이미 탈락한 학생들과 비슷한 상태**다" — 이건 타당합니다

즉 **예언 도구가 아니라 조기 경보 장치**로 쓰는 것이 맞습니다.
"휴학이 길고 성적이 낮은 상태"는 그 자체로 개입이 필요한 신호입니다.

**제대로 고치려면** 입학 첫 학기 시점의 정보(입학 성적, TOPIK, 나이, 국적)만으로 예측하거나,
학기별 시계열 데이터를 확보해 "N학기 시점 정보로 N+1학기 이탈 예측"을 해야 합니다.
현재 데이터는 학생당 한 행뿐이라 그렇게 할 수 없습니다.

### 8-2. 그 밖의 한계

| 한계 | 내용 | 영향 |
|---|---|---|
| **타깃 정의** | 자퇴 438명이 "탈락 아님"으로 학습됨 | 자퇴 위험군은 잡아내지 못합니다 |
| **표본 크기** | 대학원 제적 176명 | 변수를 더 늘리면 과적합됩니다 |
| **선형 가정** | GPA와 위험의 관계를 직선으로만 표현 | 임계점(예: 2.0 미만 급증)을 못 잡습니다 |
| **결측 변수** | 출결, 등록금 납부, 상담 이력, 기숙사 거주 없음 | 실제로 더 중요할 수 있는 요인이 빠졌습니다 |
| **집단 대표성** | 중국 국적이 78% | 소수 국적 학생에 대한 예측은 근거가 약합니다 |
| **시간 변화** | 2016~2026년을 한 덩어리로 학습 | 코로나 시기 등 제도 변화를 반영하지 못합니다 |

> **⚠️ 실무 적용 시 주의**
>
> 이 모델의 결과는 **상담 우선순위를 정하는 참고자료**입니다.
> 학사 처분이나 불이익의 근거로 쓰면 안 됩니다.
> 특정 국적·계열 학생이 위험군에 몰린다면 그것이 실제 위험인지, 데이터에 담긴 편향인지 따로 검토해야 합니다.

---

## 9. 정리

### 오늘 한 일

```
① 데이터 감사       13,090행 25열 · 결측률 확인 · 못 쓸 변수 솎아내기
② 타깃 정의         제적 vs 자퇴 — 무엇을 탈락이라 부를지 정하기
③ 집단 분리         학부 / 대학원 독립 전처리
④ LR 전용 전처리    결측 대체 → 스케일링 → 더미화 (파이프라인)
⑤ 공선성 진단       계열 ↔ 소속 대학 완전 겹침 발견 → 제거
⑥ 모델 학습         sklearn(예측) + statsmodels(해석)
⑦ 임계값            학습셋 교차검증으로 결정
⑧ 평가              PR-AUC 중심 · 정확도의 함정 확인
⑨ SHAP              전역 중요도 + 개별 학생 설명
⑩ 명단 산출         위험 요인 3가지를 붙인 CSV
```

### 핵심 교훈 5가지

1. **코드를 쓰기 전에 데이터를 세어본다.** 결측률 78% 변수, 정의된 적 없는 변수는 세어보면 바로 드러납니다.
2. **에러 없이 틀리는 코드가 가장 위험하다.** `df2`에 `df1`을 대입해도 파이썬은 아무 말 안 합니다.
3. **전처리는 모델을 따라간다.** 스케일링·더미화·결측 대체는 로지스틱 회귀라서 필요한 것입니다.
4. **불균형 데이터에서 정확도는 거짓말을 한다.** 전원 음성으로 찍어도 90%가 나옵니다.
5. **성능이 너무 좋으면 누수를 의심한다.** ROC-AUC 0.96은 칭찬이 아니라 경고였습니다.

### 다음 단계

- **XGBoost 노트북**과 비교해 보세요. 같은 데이터, 다른 모델, 다른 중요도 순위가 나옵니다.
- 8장의 한계를 하나씩 개선해 보세요. 특히 **입학 시점 정보만으로 예측**하는 모델을 만들어
  성능이 얼마나 떨어지는지 확인하면, 시점 누수의 크기를 체감할 수 있습니다.

#### ✅ 최종 체크포인트

1. 로지스틱 회귀에 스케일링이 필요한 이유를 한 문장으로 설명해 보세요.
2. 오즈비 2.5를 "위험이 2.5배"라고 말하면 왜 부정확한가요?
3. 이 모델의 ROC-AUC가 0.96인데, 이 숫자를 보고 가장 먼저 해야 할 일은?

<details><summary>답 보기</summary>

1. 모든 변수를 하나의 식에서 더하기 때문에, 단위가 다르면 큰 값을 가진 변수가 저울을 기울이고
   계수 크기를 서로 비교할 수도 없습니다.
2. 오즈비는 확률의 비가 아니라 **오즈(p/(1-p))의 비**입니다.
   사건이 드물 때만 위험비에 가까워지고, 확률이 클수록 둘의 차이가 벌어집니다.
3. **누수를 의심하고 변수의 관측 시점을 점검합니다.**
   이 경우 `이수학기수`·`최종GPA`가 제적자에게는 제적 시점 값이라 결과를 보고 원인을 맞히는 구조였습니다.

</details>

---

*이 노트북은 강의용으로 제작되었습니다. 위험군 명단은 상담 우선순위 참고자료로만 사용하세요.*